# 03. Multi-Signal Hybrid Ranking & MMR Diversity

### Objective
Understand how multi-criteria hybrid ranking balances semantic relevance with financial feasibility, seasonal weather windows, and geographical diversity.

## 1. The Composite Ranking Formulation

15919	ext{Score}(d) = \sum_{i=1}^{M} w_i \cdot S_i(d, 	ext{user})15919
where $\sum w_i = 1$, and signals include:
- {	ext{dense}}$: Semantic embedding cosine similarity
- {	ext{tfidf}}$: Lexical keyword similarity
- {	ext{pref}}$: Jaccard interest overlap
- {	ext{budget}}$: Budget compatibility curve: $\exp\left(-2.5 \cdot \max\left(0, rac{c - d}{d}ight)ight)$
- {	ext{season}}$: Month/season alignment score
- {	ext{qual}}$: Quality prior (safety & popularity)

In [ ]:
from src.ranking.scorer import HybridScorer
from src.data.models import Destination, UserPreferences
from src.retrieval.two_stage import CandidateRecord

scorer = HybridScorer()
dest = Destination(
    destination_id="EX-1", name="Zermatt", country="Switzerland", continent="Europe",
    latitude=45.9, longitude=7.7, category="Mountain", description="Alpine resort",
    est_daily_cost_inr=15000.0, best_months=[1,2,3,12]
)
# Test budget curve on budget traveler (₹30k / 7 days = ₹4.2k/day)
prefs_low = UserPreferences(budget_max_inr=30000, duration_days=7)
s_low, reason_low = scorer.score_budget_compatibility(dest, prefs_low)

# Test budget curve on luxury traveler (₹200k / 7 days = ₹28k/day)
prefs_high = UserPreferences(budget_max_inr=200000, duration_days=7)
s_high, reason_high = scorer.score_budget_compatibility(dest, prefs_high)

print(f"Budget Score for Backpacking: {s_low:.4f} -> {reason_low}")
print(f"Budget Score for Luxury:     {s_high:.4f} -> {reason_high}")

## 2. Maximal Marginal Relevance (MMR) for Diversity

15919	ext{MMR}(d_i) = \lambda \cdot 	ext{Score}(d_i) - (1 - \lambda) \cdot \max_{d_j \in S} 	ext{Sim}(d_i, d_j)15919
$\lambda = 0.75$ guarantees high topical relevance while discouraging redundant geographic or categorical duplicates.